In [1]:
import os
import shutil
import importlib.util

local_ragas_path = 'ragas'

spec = importlib.util.find_spec('ragas')
if spec is None or spec.origin is None:
    raise ImportError("Não foi possível encontrar a biblioteca 'ragas'.")

library_ragas_path = os.path.dirname(spec.origin)

if not os.path.exists(library_ragas_path):
    raise FileNotFoundError(f"O diretório da biblioteca 'ragas' não foi encontrado: {library_ragas_path}")

for root, dirs, files in os.walk(local_ragas_path):
    for file in files:
        local_file_path = os.path.join(root, file)
        relative_path = os.path.relpath(local_file_path, local_ragas_path)
        library_file_path = os.path.join(library_ragas_path, relative_path)
        os.makedirs(os.path.dirname(library_file_path), exist_ok=True)
        shutil.copy2(local_file_path, library_file_path)

print("Arquivos copiados com sucesso!")

Arquivos copiados com sucesso!


In [2]:
from llama_index.core import SimpleDirectoryReader
from ragas.testset.generator import TestsetGenerator
from ragas.testset.evolutions import simple, reasoning, multi_context, conditional
from ragas.testset.prompts import translate_prompts
from llama_index.embeddings.ollama import OllamaEmbedding
from ragas.run_config import RunConfig
from llama_index.llms.ollama import Ollama

In [3]:
data = 'data'
language = "pt"
distributions = {
    simple:0.4,
    reasoning:0.2,
    multi_context:0.2,
    conditional:0.2
    }
TIMEOUT = 2400.0
cache_dir = './cache'
amount_of_tests = 128
RUN_CONFIG = RunConfig(timeout=TIMEOUT)

In [4]:
translate_prompts(language, cache_dir)

In [5]:
embeding = OllamaEmbedding(model_name="llama3.1", request_timeout=TIMEOUT)
model = Ollama(model="llama3.1", request_timeout=TIMEOUT)

generator = TestsetGenerator.from_llama_index(
    model,
    model,
    embeding,
    run_config=RUN_CONFIG
)

In [6]:
documents = SimpleDirectoryReader(data).load_data()

In [ ]:
testset = generator.generate_with_llamaindex_docs(documents, amount_of_tests, distributions, with_debugging_logs=True, run_config=RUN_CONFIG)

embedding nodes:   0%|          | 0/174 [00:00<?, ?it/s]

Filename and doc_id are the same for all nodes.


Generating:   0%|          | 0/129 [00:00<?, ?it/s]

[ragas.testset.filters.DEBUG] context scoring: {'clarity': 2, 'depth': 1, 'structure': 2, 'relevance': 2, 'score': 1.75}
[ragas.testset.evolutions.DEBUG] keyphrases in merged node: ['Transformador Isolador', 'Proteção de Sub e Sobretensão', 'Sistema de Bypass Automático']
[ragas.testset.filters.DEBUG] context scoring: {'clarity': 3, 'depth': 2, 'structure': 3, 'relevance': 3, 'score': 2.75}
[ragas.testset.evolutions.DEBUG] keyphrases in merged node: ['Atualização de Firmware', 'Registro Permanente de Eventos Interno', 'Rede de Assistência Técnica', 'DSP Armazenamento de Registros', 'Caixa-Preta para Medidas e Alarmes']
[ragas.testset.filters.DEBUG] context scoring: {'clarity': 3, 'depth': 2, 'structure': 2, 'relevance': 1, 'score': 2.0}
[ragas.testset.evolutions.DEBUG] keyphrases in merged node: ['CM MKT OUT 23', 'Equipamento de Transformação', 'Tensão Variável', 'Potência Disponível', 'Regulação ESTATICA']
[ragas.testset.filters.DEBUG] context scoring: {'clarity': 2, 'depth': 1, 'stru

In [7]:
dataframe = testset.to_pandas()
dataframe.to_csv('testset_1.csv')